# Milestone 5 — Phase 6: Calibrated XGBoost Finalist Comparison

This notebook compares two candidates using identical five-fold outer stratified CV on the original training partition only:

- **Candidate A:** tuned unweighted XGBoost, with hyperparameters re-tuned independently inside every calibration fold before generating OOF probabilities.
- **Candidate B:** the fixed Phase 4 balanced-XGBoost configuration.

Both candidates use sigmoid calibration and a training-only F1-optimal threshold. The historical 20% holdout is permanently excluded: it is not loaded into the experiment API, scored, or used for decisions. The five outer folds are descriptive validation, not a formal significance test.

## 1. Runtime setup and dataset

Run in Google Colab. Upload the original IBM Telco CSV if it is not already present. The split is reconstructed only to isolate the original training rows; the holdout return values are immediately discarded.

In [1]:
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = "https://github.com/adhishmodi/customer-churn-prediction.git"
GIT_BRANCH = "feature/phase-6-finalist-comparison"
EXPECTED_CSV = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

current_dir = Path.cwd()
PROJECT_ROOT = next((p for p in (current_dir, *current_dir.parents) if (p / "src" / "finalist_comparison.py").is_file()), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path("/content/customer-churn-prediction")
    if (PROJECT_ROOT / ".git").is_dir():
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "fetch", "origin", GIT_BRANCH], check=True)
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "checkout", GIT_BRANCH], check=True)
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only", "origin", GIT_BRANCH], check=True)
    else:
        if PROJECT_ROOT.exists() and any(PROJECT_ROOT.iterdir()):
            raise RuntimeError(f"Existing non-repository directory: {PROJECT_ROOT}")
        subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_BRANCH, REPOSITORY_URL, str(PROJECT_ROOT)], check=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
DATA_PATH = PROJECT_ROOT / "data" / "raw" / EXPECTED_CSV
if not DATA_PATH.exists():
    from google.colab import files
    uploaded = files.upload()
    if EXPECTED_CSV not in uploaded:
        raise FileNotFoundError(f"Upload the original file named {EXPECTED_CSV}")
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    DATA_PATH.write_bytes(uploaded[EXPECTED_CSV])

print(f"Project root: {PROJECT_ROOT}")
print(f"Training source: {DATA_PATH}")

Saving WA_Fn-UseC_-Telco-Customer-Churn.csv to WA_Fn-UseC_-Telco-Customer-Churn.csv
Project root: /content/customer-churn-prediction
Training source: /content/customer-churn-prediction/data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv


In [2]:
from src.preprocess import load_and_clean_data, prepare_features_target, split_data

df = load_and_clean_data(DATA_PATH)
X, y = prepare_features_target(df)
# Reconstruct the frozen split only to isolate original training rows.
# Both historical-holdout outputs are discarded immediately and never referenced.
X_train, X_holdout, y_train, y_holdout = split_data(X, y)
del X_holdout, y_holdout
del df, X, y
print(f"Training rows: {len(X_train):,}")
print(f"Training class counts: {y_train.value_counts().sort_index().to_dict()}")

Training rows: 5,634
Training class counts: {0: 4139, 1: 1495}


## 2. Run strict nested cross-fitting

The default protocol uses five outer folds, five calibration folds within each outer-training partition, three tuning folds per randomized search, and 12 XGBoost parameter trials per search. Candidate A therefore performs nested hyperparameter searches for calibration OOF predictions and a separate search on each complete outer-training partition. This is computationally intensive; keep the protocol fixed for the primary comparison.

In [3]:
from src.finalist_comparison import compare_calibrated_xgboost_finalists

phase6_results = compare_calibrated_xgboost_finalists(
    X_train,
    y_train,
    outer_splits=5,
    calibration_splits=5,
    tuning_splits=3,
    n_iter=12,
    random_state=42,
)

assert phase6_results["configuration"]["test_set_used"] is False
assert phase6_results["configuration"]["historical_holdout_accessed"] is False
display(phase6_results["outer_fold_results"])
display(phase6_results["summary"])
display(phase6_results["thresholds_and_parameters"][["candidate", "fold", "threshold", "training_oof_f1"]])

,candidate,fold,threshold,training_oof_f1,roc_auc,average_precision,precision,recall,f1,accuracy,brier_score,log_loss
0,Tuned XGBoost (unweighted),1,0.337038,0.636529,0.844229,0.648338,0.561713,0.745819,0.640805,0.778172,0.136199,0.418756
1,Phase 4 XGBoost (balanced),1,0.313686,0.630694,0.843987,0.655543,0.541667,0.782609,0.640219,0.766637,0.135638,0.418345
2,Tuned XGBoost (unweighted),2,0.361463,0.647372,0.832839,0.637825,0.550562,0.655518,0.598473,0.766637,0.141700,0.430191
3,Phase 4 XGBoost (balanced),2,0.251067,0.643663,0.830482,0.633096,0.487705,0.795987,0.604828,0.724046,0.142158,0.430691
4,Tuned XGBoost (unweighted),3,0.275679,0.640584,0.845275,0.687928,0.517316,0.799331,0.628121,0.748891,0.133687,0.415788
5,Phase 4 XGBoost (balanced),3,0.371829,0.643426,0.844451,0.683861,0.574648,0.682274,0.623853,0.781721,0.134236,0.416367
6,Tuned XGBoost (unweighted),4,0.284641,0.635476,0.864155,0.685641,0.539326,0.802676,0.645161,0.765750,0.128302,0.394842
7,Phase 4 XGBoost (balanced),4,0.321831,0.629826,0.864761,0.687807,0.567308,0.789298,0.660140,0.784383,0.127846,0.395074
8,Tuned XGBoost (unweighted),5,0.341626,0.631028,0.862787,0.678706,0.603448,0.702341,0.649150,0.798401,0.129125,0.400450
9,Phase 4 XGBoost (balanced),5,0.362262,0.628983,0.862429,0.679545,0.604585,0.705686,0.651235,0.799290,0.128916,0.399904


,candidate,fold_count,pooled_roc_auc,pooled_average_precision,pooled_precision,pooled_recall,pooled_f1,pooled_accuracy,pooled_brier_score,pooled_log_loss,...,fold_f1_mean,fold_f1_std,fold_accuracy_mean,fold_accuracy_std,fold_brier_score_mean,fold_brier_score_std,fold_log_loss_mean,fold_log_loss_std,mean_threshold,std_threshold
0,Tuned XGBoost (unweighted),5,0.848896,0.663655,0.551793,0.741137,0.632601,0.771565,0.133803,0.412007,...,0.632342,0.020514,0.771570,0.018276,0.133803,0.005483,0.412005,0.014307,0.320089,0.037722
1,Phase 4 XGBoost (balanced),5,0.848748,0.664414,0.550490,0.751171,0.635361,0.771211,0.133760,0.412078,...,0.636055,0.022090,0.771215,0.028800,0.133759,0.005760,0.412076,0.014504,0.324135,0.047911
2,Paired fold delta (Tuned XGBoost (unweighted) ...,5,0.000148,-0.000759,0.001303,-0.010033,-0.002760,0.000355,0.000044,-0.000071,...,-0.003713,0.007395,0.000355,0.029036,0.000044,0.000517,-0.000071,0.000520,-0.004046,NaN


,candidate,fold,threshold,training_oof_f1
0,Tuned XGBoost (unweighted),1,0.337038,0.636529
1,Phase 4 XGBoost (balanced),1,0.313686,0.630694
2,Tuned XGBoost (unweighted),2,0.361463,0.647372
3,Phase 4 XGBoost (balanced),2,0.251067,0.643663
4,Tuned XGBoost (unweighted),3,0.275679,0.640584
5,Phase 4 XGBoost (balanced),3,0.371829,0.643426
6,Tuned XGBoost (unweighted),4,0.284641,0.635476
7,Phase 4 XGBoost (balanced),4,0.321831,0.629826
8,Tuned XGBoost (unweighted),5,0.341626,0.631028
9,Phase 4 XGBoost (balanced),5,0.362262,0.628983


## 3. Export validation evidence

Upload the four CSV files for review. The paired fold deltas are descriptive only; five outer folds do not establish statistical significance. Do not evaluate or recover the historical holdout for this phase.

In [4]:
from src.config import REPORTS_DIR

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
outputs = {
    "phase_6_outer_fold_results.csv": phase6_results["outer_fold_results"],
    "phase_6_fold_predictions.csv": phase6_results["fold_predictions"],
    "phase_6_summary.csv": phase6_results["summary"],
    "phase_6_thresholds_and_parameters.csv": phase6_results["thresholds_and_parameters"],
}
for filename, frame in outputs.items():
    frame.to_csv(REPORTS_DIR / filename, index=False)
    print(f"Saved: {REPORTS_DIR / filename}")
print("Phase 6 complete. Historical holdout was not accessed.")

Saved: /content/customer-churn-prediction/reports/phase_6_outer_fold_results.csv
Saved: /content/customer-churn-prediction/reports/phase_6_fold_predictions.csv
Saved: /content/customer-churn-prediction/reports/phase_6_summary.csv
Saved: /content/customer-churn-prediction/reports/phase_6_thresholds_and_parameters.csv
Phase 6 complete. Historical holdout was not accessed.


## 4. Completion checklist

- [ ] Run every code cell successfully in Colab.
- [ ] Confirm five outer folds for both candidates and identical validation rows per fold.
- [ ] Confirm each candidate has exactly one outer-fold prediction for every training row.
- [ ] Review fold-level F1, paired fold deltas, ROC-AUC, Average Precision, precision, recall, accuracy, Brier score, and log loss.
- [ ] Review fold-specific thresholds and Candidate A hyperparameter stability.
- [ ] Upload all four CSV files for review.
- [ ] Record the conclusion without using the historical holdout.

**Interpretation rule:** use mean outer-fold F1 as the primary comparison, supported by paired fold differences and calibration metrics. Treat all five-fold summaries as descriptive; do not claim statistical significance.